# Phase 4: SFT — Fine-Tune Qwen3.5 for Tool Calling

Supervised fine-tuning on the teacher trajectories from Phase 2.
Teaches the model: which tools to call, what arguments to pass, when to stop, and how to write the final snapshot.

**Training:** Use `_phase_4_sft_colab.ipynb` on Google Colab (free T4 GPU). MLX LoRA backward pass
needs ~48 GB regardless of model size — infeasible on 16GB Mac.

**This notebook:** Data prep, config generation (for reference), and **evaluation** of the fine-tuned model
served locally via `mlx_lm.server`.

**Workflow:** Colab trains → push to HF Hub → `mlx_lm.convert` → serve locally → eval cells below.

## MLX SFT (Mac)

Train locally on Apple Silicon using `mlx_lm.lora`. This is the same LoRA approach as Unsloth
but runs on Metal instead of CUDA.

**Model choice:** `Qwen3.5-2B` instead of 4B. On 16GB Mac, 4B OOMs during backward pass at
any useful sequence length. 2B trains comfortably at 16K context with full LoRA — no frozen laptop.

**Key differences from Databricks path:**

| | MLX (Mac) | Unsloth (Databricks) |
|---|---|---|
| Base model | `mlx-community/Qwen3.5-2B-4bit` | `unsloth/Qwen3-4B` |
| LoRA config | YAML: `rank: 32, scale: 2.0` | Python: `r=32, lora_alpha=64` |
| Masking | `--mask-prompt` flag | `train_on_responses_only()` |
| Batch size | 1 (16GB memory) | 4 (GPU VRAM) |
| Sequence length | 16384 (2B model fits) | 8192 |

**Important:**
- MLX uses `scale = alpha / rank`. So `rank=32, scale=2.0` → `alpha=64`.
- Qwen3.5's Jinja template expects tool call `arguments` as a dict, not a JSON string. The data prep cell handles this.
- `grad_checkpoint=true` is always recommended on 16GB Mac.

In [1]:
import json
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS
from bbb.helpers__data_gen import SYSTEM_PROMPT, TICKERS, make_user_prompt
from bbb.helpers__inference import compute_reward
from bbb.agent import run_tool_calling_agent_chat, TOOL_SCHEMAS_CHAT

VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

### Data Prep

MLX expects `train.jsonl` and `valid.jsonl` in a data directory. Each line is a JSON object
with `messages` and `tools` keys — exactly what `trajectories_sft.jsonl` already contains.

We split 85/15 and save to `data/bbb/mlx_sft/`.

In [2]:
# Load SFT trajectories
sft_path = PROJECT_ROOT / "data" / "bbb" / "trajectories_sft.jsonl"
records = []
with open(sft_path) as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} trajectories")

# Inspect structure
for i, r in enumerate(records[:3]):
    roles = [m["role"] for m in r["messages"]]
    has_think = any("<think>" in (m.get("content", "") or "") for m in r["messages"])
    print(f"  [{i}] {r.get('ticker','?')} — {len(r['messages'])} msgs, {len(r['tools'])} tools, think={has_think}")
    print(f"       roles: {roles}")

Loaded 10 trajectories
  [0] WDAY — 9 msgs, 4 tools, think=True
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']
  [1] DUK — 10 msgs, 4 tools, think=True
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']
  [2] LLY — 9 msgs, 4 tools, think=True
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']


In [3]:
# Split and save for MLX
random.seed(42)
random.shuffle(records)


def to_mlx_format(record: dict) -> dict:
    """
    Convert SFT record to MLX-compatible format.

    Key fix: Qwen3.5's Jinja template uses `arguments | items` which expects a dict,
    but responses_to_hermes() stores arguments as a JSON string. Parse it here.
    """
    messages = []
    for m in record["messages"]:
        m = dict(m)  # shallow copy
        if m.get("tool_calls"):
            m["tool_calls"] = [
                {
                    **tc,
                    "function": {
                        **tc["function"],
                        "arguments": (
                            json.loads(tc["function"]["arguments"])
                            if isinstance(tc["function"]["arguments"], str)
                            else tc["function"]["arguments"]
                        ),
                    },
                }
                for tc in m["tool_calls"]
            ]
        messages.append(m)
    return {"messages": messages, "tools": record["tools"]}


split_idx = int(len(records) * 0.85)
train_records = records[:split_idx]
eval_records = records[split_idx:]

# Ensure at least 1 eval sample
if not eval_records:
    eval_records = [train_records.pop()]

mlx_data_dir = PROJECT_ROOT / "data" / "bbb" / "mlx_sft"
mlx_data_dir.mkdir(parents=True, exist_ok=True)

for split_name, split_data in [("train", train_records), ("valid", eval_records)]:
    path = mlx_data_dir / f"{split_name}.jsonl"
    with open(path, "w") as f:
        for r in split_data:
            f.write(json.dumps(to_mlx_format(r)) + "\n")
    print(f"Saved {len(split_data)} records to {path}")

train_tickers = {r.get("ticker") for r in train_records}
print(f"\nTrain tickers: {train_tickers}")

Saved 8 records to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/mlx_sft/train.jsonl
Saved 2 records to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/mlx_sft/valid.jsonl

Train tickers: {'NKE', 'LLY', 'AMD', 'HSY', 'DHR', 'BA', 'BABA', 'ECL'}


### Training Config

Generate the YAML config for `mlx_lm.lora`. Key settings:
- `mask_prompt: true` — only train on assistant turns (same as Unsloth's `train_on_responses_only`)
- `grad_checkpoint: true` — trades ~30% speed for lower memory (required on 16GB Mac)
- `rank: 16, scale: 2.0` — equivalent to `alpha=32`
- `learning_rate: 1e-5` — MLX default, lower than Unsloth's 2e-4 (MLX uses full-precision Adam)
- `keys: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]` — **critical!** MLX's default LoRA for Qwen3 only targets q_proj + v_proj. Specifying all 7 projection layers dramatically increases trainable parameters and learning capacity.

**16GB Mac gotcha:** `warmup_steps` is NOT a valid `mlx_lm.lora` config key — it's silently ignored. Use `lr_schedule` for warmup if needed.

In [4]:
import yaml

MLX_MODEL = "mlx-community/Qwen3.5-2B-4bit"

config = {
    # Model
    "model": MLX_MODEL,
    "train": True,
    "data": str(mlx_data_dir),
    # Training
    "iters": 20,
    "batch_size": 1,
    "grad_accumulation_steps": 4,
    "max_seq_length": 8192,
    "mask_prompt": True,
    "grad_checkpoint": True,
    "learning_rate": 1e-5,
    # LoRA — target all projection layers (default Qwen3 only hits q_proj + v_proj)
    # See: https://github.com/ml-explore/mlx/issues/2616
    "lora_parameters": {
        "rank": 16,
        "scale": 2.0,       # = alpha / rank
        "dropout": 0.0,
        "keys": [
            "self_attn.q_proj",
            "self_attn.k_proj",
            "self_attn.v_proj",
            "self_attn.o_proj",
            "mlp.gate_proj",
            "mlp.up_proj",
            "mlp.down_proj",
        ],
    },
    "num_layers": 16,
    # Checkpoints & logging
    "adapter_path": str(PROJECT_ROOT / "models" / "mlx_sft_adapters"),
    "save_every": 25,
    "steps_per_report": 5,
    "steps_per_eval": 25,
}

config_path = PROJECT_ROOT / "nb" / "bbb" / "sft_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to {config_path}")
print(f"\n{yaml.dump(config, default_flow_style=False, sort_keys=False)}")

Config saved to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/nb/bbb/sft_config.yaml

model: mlx-community/Qwen3.5-2B-4bit
train: true
data: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/mlx_sft
iters: 20
batch_size: 1
grad_accumulation_steps: 4
max_seq_length: 8192
mask_prompt: true
grad_checkpoint: true
learning_rate: 1.0e-05
lora_parameters:
  rank: 16
  scale: 2.0
  dropout: 0.0
  keys:
  - self_attn.q_proj
  - self_attn.k_proj
  - self_attn.v_proj
  - self_attn.o_proj
  - mlp.gate_proj
  - mlp.up_proj
  - mlp.down_proj
num_layers: 16
adapter_path: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/models/mlx_sft_adapters
save_every: 25
steps_per_report: 5
steps_per_eval: 25



### Train

Run `mlx_lm.lora` with the config. Run in a separate terminal:

```bash
# On 16GB Mac, disable JIT compilation to avoid backward-pass memory spike
# See: https://github.com/ml-explore/mlx-lm/issues/516
MLX_DISABLE_COMPILE=1 uv run mlx_lm.lora --config nb/bbb/sft_config.yaml
```

**Why `MLX_DISABLE_COMPILE=1`?** The trainer wraps the training step in `mx.compile()` for speed.
The first call traces the entire forward+backward graph into memory at once. On 16GB Mac with
other apps running (Cursor, Chrome), this spike exceeds available Metal memory → silent SIGKILL.
Disabling compile runs ~30-40% slower per iteration but avoids the memory spike.

**Before training:** Close Chrome and other memory-heavy apps. On unified memory Macs,
GPU and CPU share the same 16GB — every GB matters.

In [5]:
#!uv run mlx_lm.lora --config {config_path}

### Fuse Adapters

Merge the LoRA weights into the base model to create a standalone model directory.
This fused model can be served directly via `mlx_lm.server`.

In [ ]:
adapter_path = PROJECT_ROOT / "models" / "mlx_sft_adapters"
fused_path = PROJECT_ROOT / "models" / "mlx_sft_fused"

!uv run mlx_lm.fuse \
    --model {MLX_MODEL} \
    --adapter-path {adapter_path} \
    --save-path {fused_path}

print(f"Fused model saved to {fused_path}")

### Evaluate SFT Model

Serve the fused model and run the same agent loop from Phase 3 on unseen tickers.

**Start the server in a terminal:**
```bash
uv run mlx_lm.server --model models/mlx_sft_fused --port 8080 --prompt-cache-size 4
```

Then run the cells below.

In [ ]:
from openai import AsyncOpenAI

sft_client = AsyncOpenAI(base_url="http://localhost:8080/v1", api_key="none")

# Warm up cache
warmup = await sft_client.chat.completions.create(
    model="default_model",
    messages=[{"role": "user", "content": "Say hello."}],
    max_tokens=10,
)
print(f"Cache warmed: {warmup.choices[0].message.content!r}")

In [ ]:
import asyncio
import time
from tqdm.asyncio import tqdm_asyncio

unseen_tickers = [t for t in TICKERS if t not in train_tickers][:10]
print(f"Testing on {len(unseen_tickers)} unseen tickers: {unseen_tickers}")

semaphore = asyncio.Semaphore(3)


async def eval_one(ticker: str, focus: str) -> dict:
    async with semaphore:
        try:
            res = await run_tool_calling_agent_chat(
                client=sft_client,
                model="default_model",
                user_prompt=make_user_prompt(ticker, focus),
                system_prompt=SYSTEM_PROMPT,
                tools=TOOL_SCHEMAS_CHAT,
                tool_functions=TOOL_FUNCTIONS,
                max_iterations=5,
                max_tokens=4096,
                verbose=False,
            )
            n_tool_calls = sum(len(m.get("tool_calls", [])) for m in res["input"])
            reward = compute_reward(res["input"], VALID_TOOL_NAMES, res.get("reasoning_summaries"))
            return {
                "ticker": ticker, "focus": focus,
                "n_tool_calls": n_tool_calls,
                "output_len": len(res["output_text"]),
                "reward": reward,
            }
        except Exception as e:
            return {"ticker": ticker, "focus": focus, "error": str(e), "reward": {"total": -3.0}}


focuses = ["financial health", "growth potential", "competitive position"]
tasks = [eval_one(t, random.choice(focuses)) for t in unseen_tickers]

t0 = time.time()
sft_results = await tqdm_asyncio.gather(*tasks, desc="SFT eval")
elapsed = time.time() - t0

errors = [r for r in sft_results if "error" in r]
print(f"\nDone: {len(sft_results)} tickers in {elapsed:.1f}s ({elapsed/len(sft_results):.1f}s/ticker)")
if errors:
    for e in errors:
        print(f"  {e['ticker']}: {e['error']}")

In [ ]:
# Compare SFT vs baseline
sft_rewards = [r["reward"]["total"] for r in sft_results if isinstance(r["reward"], dict)]
sft_successful = [r for r in sft_results if isinstance(r["reward"], dict) and "total" in r["reward"]]

print(f"=== SFT MODEL ({len(sft_successful)}/{len(sft_results)} successful) ===")
print(f"Reward:       avg={sum(sft_rewards)/len(sft_rewards):.2f}  min={min(sft_rewards):.2f}  max={max(sft_rewards):.2f}")

components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
print("\n--- Per-component averages ---")
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in sft_successful]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

# Load baseline for comparison
baseline_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results_mlx.jsonl"
if baseline_path.exists():
    baseline_rewards = []
    with open(baseline_path) as f:
        for line in f:
            r = json.loads(line)
            if isinstance(r.get("reward"), dict):
                baseline_rewards.append(r["reward"]["total"])

    print(f"\n{'Metric':<20} {'Baseline':>10} {'SFT':>10} {'Delta':>10}")
    print("-" * 55)
    b_avg = sum(baseline_rewards) / len(baseline_rewards)
    s_avg = sum(sft_rewards) / len(sft_rewards)
    print(f"{'Avg Reward':<20} {b_avg:>+10.2f} {s_avg:>+10.2f} {s_avg - b_avg:>+10.2f}")
else:
    print("\nNo baseline results found — run Phase 3 batch first for comparison")